# Notebook 1 — Data Ingestion & Cleaning

**Goal:** Load the Spotify dataset, understand its shape, fix issues, and save a clean version.

**Dataset:** 114,000 Spotify tracks with 21 audio features from Kaggle.

In [ ]:
import pandas as pd
import numpy as np
import os

print('Libraries loaded ✓')

## 1. Download the dataset

Run this once. You need a free Kaggle account and your `kaggle.json` API key in `~/.kaggle/`.

Get your key at: https://www.kaggle.com/settings → API → Create New Token

In [ ]:
import subprocess

os.makedirs('../data', exist_ok=True)

if not os.path.exists('../data/dataset.csv'):
    print('Downloading dataset...')
    subprocess.run([
        'kaggle', 'datasets', 'download',
        'maharshipandya/-spotify-tracks-dataset',
        '--unzip', '-p', '../data'
    ])
    print('Download complete ✓')
else:
    print('Dataset already exists ✓')

## 2. Load and inspect

In [ ]:
df = pd.read_csv('../data/dataset.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# Basic info — dtypes and null counts
df.info()

In [ ]:
# Summary statistics — get a feel for distributions
df.describe().round(3)

## 3. Check for issues

In [ ]:
# Missing values
missing = df.isnull().sum()
missing[missing > 0]

In [ ]:
# Duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate track IDs: {df["track_id"].duplicated().sum()}')

In [ ]:
# Target variable distribution
print('Popularity distribution:')
print(df['popularity'].describe())
print(f'\nZero-popularity tracks: {(df["popularity"] == 0).sum()} ({(df["popularity"] == 0).mean():.1%})')

## 4. Clean

In [ ]:
df_clean = df.copy()

# Drop exact duplicates
df_clean = df_clean.drop_duplicates(subset='track_id')

# Drop tracks with zero popularity — likely unlisted/unreleased
df_clean = df_clean[df_clean['popularity'] > 0]

# Drop rows with nulls in audio features
audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_ms'
]
df_clean = df_clean.dropna(subset=audio_features)

# Convert duration from ms to seconds (more interpretable)
df_clean['duration_s'] = (df_clean['duration_ms'] / 1000).round(1)

# Create binary target: 1 = hit (top 25% popularity), 0 = not
threshold = df_clean['popularity'].quantile(0.75)
df_clean['is_hit'] = (df_clean['popularity'] >= threshold).astype(int)

print(f'Clean dataset shape: {df_clean.shape}')
print(f'Hit threshold (top 25%): popularity >= {threshold:.0f}')
print(f'Class balance — hits: {df_clean["is_hit"].mean():.1%}')

## 5. Save clean data

In [ ]:
df_clean.to_csv('../data/spotify_clean.csv', index=False)
print(f'Saved → data/spotify_clean.csv ({df_clean.shape[0]:,} rows)')

---
## Summary

| Step | Result |
|---|---|
| Raw rows | 114,000 |
| After dedup + filter | ~89,000 |
| Target | `is_hit` (binary, 25% positive) |
| Features | 10 audio features + genre + explicit |

**Next:** `02_eda.ipynb` — explore what separates hits from non-hits.